# Demo — Run an Episode in `openenv-r2-kit`

This notebook walks through the environment end-to-end without any training:

1. Install the template
2. Reset → inspect the initial observation
3. Take a few actions, watch the reward breakdown
4. Compare random vs heuristic policies across a few episodes

**Colab-friendly:** runs on CPU in under a minute. No GPU required.

## 1. Install the template

In [ ]:
import os
REPO = 'openenv-r2-kit'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Kaviyamurugadass/openenv-r2-kit.git
%cd {REPO}
!pip install -q uv
!uv sync --quiet

## 2. Reset and inspect the starting observation

In [ ]:
import json
import sys; sys.path.insert(0, '.')

from server.environment import Environment
from models import Action, ActionCommand

env = Environment(domain_randomise=False, seed=42)
obs = env.reset(scenario_name='placeholder_easy')

print(json.dumps(obs.model_dump(), indent=2))

## 3. Take one step — see the decomposed reward

Key signal: `reward_breakdown` explains *why* the reward is what it is. Judges look at this.

In [ ]:
obs = env.step(Action(
    command=ActionCommand.INSPECT,
    args={'target': 'example_file.py'},
    justification='Examining before acting',
    confidence=0.8,
))

print(f'reward      = {obs.reward:.3f}')
print(f'breakdown   = {json.dumps(obs.reward_breakdown, indent=2)}')
print(f'violations  = {obs.rule_violations}')
print(f'step        = {obs.step}/{obs.max_steps}')

## 4. Run a full episode to termination

In [ ]:
env = Environment(domain_randomise=False, seed=42)
obs = env.reset(scenario_name='placeholder_medium')
rewards = []
while not (obs.done or obs.truncated):
    obs = env.step(Action(command=ActionCommand.NOOP))
    rewards.append(obs.reward)

print(f'episode ended in {obs.step} steps')
print(f'per-step rewards: {[round(r, 3) for r in rewards]}')
print(f'cumulative:       {sum(rewards):.3f}')

## 5. Compare random vs heuristic policies

These are the two policies the template ships with. Real training (next notebook) will beat both.

In [ ]:
from training.rollout_collection import collect
from training.evaluation import EvaluationSuite

for policy in ['random', 'heuristic']:
    ds = collect(
        episodes=5,
        policy=policy,
        output_dir=f'/tmp/demo_{policy}',
        seed=7,
        domain_randomise=True,
    )
    metrics = EvaluationSuite.online_metrics(ds)
    print(f'\n--- {policy} ---')
    print(f'  mean_reward    = {metrics["mean_reward"]:.3f}')
    print(f'  success_rate   = {metrics["success_rate"]:.1%}')
    print(f'  mean_length    = {metrics["mean_length"]:.1f}')

## 6. Full state — debug view (not what the agent sees)

In POMDP design, agents receive `Observation` (projected view). `State` holds hidden ground truth.

In [ ]:
print(json.dumps(env.state.model_dump(), indent=2, default=str))

## Next steps

- **Train an agent:** see `02_train_grpo.ipynb`
- **Evaluate before/after:** see `03_evaluate.ipynb`
- **Specialize to your domain:** see `FINALE_GUIDE.md` (the 8-step fill)